In [1]:
%run 0_1_load_paths.ipynb

In [2]:
import json
import os.path

import commute_dm.utils
import credentials
import momapy_kb.lpg.backends.neo4j
import momapy_kb.lpg.session

In [3]:
backend = momapy_kb.lpg.backends.neo4j.Neo4jBackend(
    hostname=credentials.NEO4J_URI,
    username=credentials.NEO4J_USERNAME,
    password=credentials.NEO4J_PASSWORD,
    notifications_min_severity="off",
)
session = momapy_kb.lpg.session.Session(backend)

In [4]:
def get_covid_pd_interface(session):
    query = """
        MATCH
            (collection:Collection)-[:HAS_ENTRY]->(collection_entry:CollectionEntry),
            (collection_entry)-[:HAS_ELEMENT_TO_ANNOTATIONS]->(annotations:Mapping),
            (collection_entry)-[:HAS_ID_TO_ELEMENT]->(ids:Mapping),
            (collection_entry)-[:HAS_OBJ]->(map:CellDesignerMap)-[:HAS_MODEL]->(model:CellDesignerModel),
            (annotations)-[:HAS_ITEM]->(annotations_item:Item),
            (annotations_item)-[:HAS_KEY]->(model_element:CellDesignerModelElement),
            (annotations_item)-[:HAS_VALUE]->(annotations_bag:Bag),
            (annotations_bag)-[:HAS_ITEM]->(annotation:RDFAnnotation),
            (annotation)-[:HAS_QUALIFIER]->(qualifier_node:BQBiol),
            (ids)-[:HAS_ITEM]->(ids_item:Item),
            (ids_item)-[:HAS_KEY]->(id:String),
            (ids_item)-[:HAS_VALUE]->(model_element)
        UNWIND annotation.resources AS resource
        WITH
            qualifier_node.value AS qualifier,
            resource AS resource,
            collect(DISTINCT collection.name) AS collection_names,
            collect([collection.name, collection_entry.file_path, id.value, model_element.name]) AS elements
        WHERE
            NOT resource CONTAINS "pubmed"
            AND NOT resource CONTAINS "doi"
            AND SIZE(collection_names) >= 2
        RETURN collect([qualifier, resource]) AS annotations, elements
    """
    results = session.execute_query(query)
    return results


def write_covid_pd_interface_to_json_file(interface_results, output_file_path):
    description = (
        "Interface between the COVID and PD maps, based on annotations.\n"
        "This file is generated by the get_interfaces notebook.\n"
        "The interface is formed of all sets of elements (species, processes, or compartments) of the COVID and PD maps that share at least\n"
        "one annotation that is not a publication, and that include at least one element from a COVID map and one from a PD map."
    )
    interface = []
    for row in interface_results:
        interface.append(
            {
                "annotations": [
                    {"qualifier": annotation[0], "resources": annotation[1]}
                    for annotation in row["annotations"]
                ],
                "model_elements": [
                    {
                        "collection": shared_element[0],
                        "map_file_or_subgraph": shared_element[1],
                        "model_element_id": shared_element[2],
                        "model_element_name": shared_element[3],
                    }
                    for shared_element in row["elements"]
                ],
            }
        )
    output = {"description": description, "data": interface}
    with open(output_file_path, "w") as f:
        json.dump(output, f)


def get_covid_ad_interface(session):
    query = """
       CALL () {
            MATCH
                (covid_collection:Collection {name: "COVID_DM_CD"}),
                (covid_collection)-[:HAS_ENTRY]->(covid_entry:CollectionEntry),
                (covid_entry)-[:HAS_ELEMENT_TO_ANNOTATIONS]->(covid_annotations:Mapping),
                (covid_entry)-[:HAS_OBJ]->(covid_map:CellDesignerMap)-[:HAS_MODEL]->(covid_model:CellDesignerModel),
                (covid_entry)-[:HAS_ID_TO_ELEMENT]->(covid_ids:Mapping),
                (covid_annotations)-[:HAS_ITEM]->(covid_annotations_item:Item),
                (covid_annotations_item)-[:HAS_KEY]->(covid_protein:Protein),
                (covid_annotations_item)-[:HAS_VALUE]->(covid_annotations_bag:Bag),
                (covid_annotations_bag)-[:HAS_ITEM]->(covid_annotation:RDFAnnotation),
                (covid_ids)-[:HAS_ITEM]->(covid_ids_item:Item),
                (covid_ids_item)-[:HAS_KEY]->(covid_id:String),
                (covid_ids_item)-[:HAS_VALUE]->(covid_protein)
            WITH
                split(covid_annotation.resources[0], ":")[2] AS covid_namespace,
                split(covid_annotation.resources[0], ":")[-1] AS covid_identifier,
                covid_collection AS covid_collection,
                covid_entry AS covid_entry,
                covid_protein AS covid_protein,
                covid_id AS covid_id
            WHERE
                covid_namespace = "hgnc.symbol"
            RETURN
                "HGNC" AS namespace, covid_identifier AS identifier, covid_collection.name AS collection_name, covid_entry.file_path AS model_id, covid_protein.name AS protein_name, covid_id.value AS protein_id
            UNION
            MATCH
                (ad_collection:Collection {name: "AD_KG_BEL"}),
                (ad_collection)-[:HAS_ENTRY]->(ad_entry:CollectionEntry),
                (ad_entry)-[:HAS_OBJ]->(ad_model:BELModel),
                (ad_model)-[:HAS_SUBGRAPH]->(ad_subgraph),
                (ad_subgraph)-[:HAS_NODE]->(ad_protein:Protein)
            WHERE
                ad_protein.namespace = "HGNC"
            RETURN
                ad_protein.namespace AS namespace, ad_protein.name AS identifier, ad_collection.name AS collection_name, ad_subgraph.name AS model_id, ad_protein.name AS protein_name, ad_protein.bel AS protein_id
        }
        WITH
            namespace AS namespace,
            identifier AS identifier,
            collect(DISTINCT [collection_name]) AS collection_names,
            collect(DISTINCT [collection_name, model_id, protein_id, protein_name]) AS collection_names_model_ids_protein_ids_protein_names
        WHERE
            size(collection_names) >= 2
        RETURN
            collect([namespace, identifier]) AS annotations, collection_names_model_ids_protein_ids_protein_names AS elements
    """
    results = session.execute_query(query)
    return results


def write_covid_ad_interface_to_json_file(interface_results, output_file_path):
    description = (
        "Interface between the COVID maps and the AD KG (from Fraunhofer), based on HGNC symbols (80% of proteins in the AD KG are identifier whith such a symbol).\n"
        "This file is generated by the get_interfaces notebook.\n"
        "The interface is formed of all proteins of the AD KG that have an HGNC based identifier and for which there is at least one protein in the COVID maps\n"
        "having an annotation with the same HGNC identifier."
    )
    interface = []
    for row in interface_results:
        interface.append(
            {
                "annotations": [
                    {"namespace": annotation[0], "identifier": annotation[1]}
                    for annotation in row["annotations"]
                ],
                "model_elements": [
                    {
                        "collection": shared_element[0],
                        "map_file_or_subgraph": shared_element[1],
                        "model_element_id": shared_element[2],
                        "model_element_name": shared_element[3],
                    }
                    for shared_element in row["elements"]
                ],
            }
        )
    output = {"description": description, "data": interface}
    with open(output_file_path, "w") as f:
        json.dump(output, f)


def get_covid_pd_ad_interface(session):
    query = """
    CALL () {
      MATCH
          (covid_collection:Collection),
          (covid_collection)-[:HAS_ENTRY]->(covid_entry:CollectionEntry),
          (covid_entry)-[:HAS_ELEMENT_TO_ANNOTATIONS]->(covid_annotations:Mapping),
          (covid_entry)-[:HAS_OBJ]->(covid_map:CellDesignerMap)-[:HAS_MODEL]->(covid_model:CellDesignerModel),
          (covid_entry)-[:HAS_ID_TO_ELEMENT]->(covid_ids:Mapping),
          (covid_annotations)-[:HAS_ITEM]->(covid_annotations_item:Item),
          (covid_annotations_item)-[:HAS_KEY]->(covid_protein:Protein),
          (covid_annotations_item)-[:HAS_VALUE]->(covid_annotations_bag:Bag),
          (covid_annotations_bag)-[:HAS_ITEM]->(covid_annotation:RDFAnnotation),
          (covid_ids)-[:HAS_ITEM]->(covid_ids_item:Item),
          (covid_ids_item)-[:HAS_KEY]->(covid_id:String),
          (covid_ids_item)-[:HAS_VALUE]->(covid_protein)
      WITH
          split(covid_annotation.resources[0], ":")[2] AS covid_namespace,
          split(covid_annotation.resources[0], ":")[-1] AS covid_identifier,
          covid_collection AS covid_collection,
          covid_entry AS covid_entry,
          covid_protein AS covid_protein,
          covid_id AS covid_id
      WHERE
          covid_namespace = "hgnc.symbol"
      RETURN
          "HGNC" AS namespace, covid_identifier AS identifier, covid_collection.name AS collection_name, covid_entry.file_path AS model_id, covid_protein.name AS protein_name, covid_id.value AS protein_id
      UNION
      MATCH
          (ad_collection:Collection {name: "AD_KG_BEL"}),
          (ad_collection)-[:HAS_ENTRY]->(ad_entry:CollectionEntry),
          (ad_entry)-[:HAS_OBJ]->(ad_model:BELModel),
          (ad_model)-[:HAS_SUBGRAPH]->(ad_subgraph),
          (ad_subgraph)-[:HAS_NODE]->(ad_protein:Protein)
      WHERE
          ad_protein.namespace = "HGNC"
      RETURN
          ad_protein.namespace AS namespace, ad_protein.name AS identifier, ad_collection.name AS collection_name, ad_subgraph.name AS model_id, ad_protein.name AS protein_name, ad_protein.bel AS protein_id
      }
      WITH
          namespace AS namespace,
          identifier AS identifier,
          collect(DISTINCT [collection_name]) AS collection_names,
          collect(DISTINCT [collection_name, model_id, protein_id, protein_name]) AS collection_names_model_ids_protein_ids_protein_names
      WHERE
          size(collection_names) >= 3
      RETURN
          collect([namespace, identifier]) AS annotations, collection_names_model_ids_protein_ids_protein_names AS elements
    """
    results = session.execute_query(query)
    return results


def write_covid_pd_ad_interface_to_json_file(interface_results, output_file_path):
    description = (
        "Interface between the COVID maps, PD maps, and the AD KG (from Fraunhofer), based on HGNC symbols (80% of proteins in the AD KG are identifier whith such a symbol).\n"
        "This file is generated by the get_interfaces notebook.\n"
        "The interface is formed of all proteins of the AD KG that have an HGNC based identifier and for which there is at least one protein in the COVID maps\n"
        "and one protein in the PD maps having an annotation with the same HGNC identifier."
    )
    interface = []
    for row in interface_results:
        interface.append(
            {
                "annotations": [
                    {"namespace": annotation[0], "identifier": annotation[1]}
                    for annotation in row["annotations"]
                ],
                "model_elements": [
                    {
                        "collection": shared_element[0],
                        "map_file_or_subgraph": shared_element[1],
                        "model_element_id": shared_element[2],
                        "model_element_name": shared_element[3],
                    }
                    for shared_element in row["elements"]
                ],
            }
        )
    output = {"description": description, "data": interface}
    with open(output_file_path, "w") as f:
        json.dump(output, f)

In [5]:
COVID_PD_INTERFACE_FILE = os.path.join(INTERFACE_DIR, "covid_pd.json")
COVID_AD_INTERFACE_FILE = os.path.join(INTERFACE_DIR, "covid_ad.json")
COVID_PD_AD_INTERFACE_FILE = os.path.join(INTERFACE_DIR, "covid_pd_ad.json")

We remake the directory where we store the interfaces:

In [6]:
commute_dm.utils.remake_dir(INTERFACE_DIR)

## Computing the interface between the maps based on annotations

### Interface between the COVID DM CD and the PD DM CD

We compute the interface between the COVID and PD maps based on annotations, i.e., we query all sets of (_collection_, _model_, _model element_) triples such that:
- _model element_ belongs to _model_, and _model_ belongs to _collection_ (COVID or PD);
- all _model elements_ of the set share at least one _annotation_ (same _qualifier_ and _resource_) that is not a pubmed annotation and;
- there is at least one _model element_ of the set belonging to a COVID _model_, and one belonging to a PD _model_.

In [7]:
interface_results = get_covid_pd_interface(session)

We save the interface to a JSON file:

In [8]:
write_covid_pd_interface_to_json_file(interface_results, COVID_PD_INTERFACE_FILE)

### Interface between the COVID DM CD and the AD KG BEL

We compute the interface between the COVID and PD maps based on annotations, i.e., we query all sets of (_collection_, _model_, _model element_) triples such that:
- _model element_ belongs to _model_, and _model_ belongs to _collection_ (COVID or PD);
- all _model elements_ of the set share at least one _annotation_ (same _qualifier_ and _resource_) that is not a pubmed annotation and;
- there is at least one _model element_ of the set belonging to a COVID _model_, and one belonging to a PD _model_.

In [9]:
interface_results = get_covid_ad_interface(session)

We save the interface to a JSON file:

In [10]:
write_covid_ad_interface_to_json_file(interface_results, COVID_AD_INTERFACE_FILE)

### Interface between the COVID DM CD, the PD DM CD, and the AD KG BEL

We compute the interface between the COVID and PD maps based on annotations, i.e., we query all sets of (_collection_, _model_, _model element_) triples such that:
- _model element_ belongs to _model_, and _model_ belongs to _collection_ (COVID or PD);
- all _model elements_ of the set share at least one _annotation_ (same _qualifier_ and _resource_) that is not a pubmed annotation and;
- there is at least one _model element_ of the set belonging to a COVID _model_, and one belonging to a PD _model_.

In [11]:
interface_results = get_covid_pd_ad_interface(session)

We save the interface to a JSON file:

In [12]:
write_covid_pd_ad_interface_to_json_file(interface_results, COVID_PD_AD_INTERFACE_FILE)